### Referencing

The dataset has like ~ 500 unique references (papers, patents). We need to cite them all.

For papers, the get_ris_from_dois function is going to do most of the heavy lifting for us. Using requests, we can use the doi.org lookup to return a .ris compatible set of data which we can _import directly into endnote_ which is just wonderful. wow.

For patents, its not quite as simple. We need to first get a unique list of patent numbers. Using lens.org we can manually enter these like 20 or 30 at a time like this:

* Search "WO2015190399A1 OR TWI629343B OR WO2020026583A1"
* Select matching patents (NOTE they might have different numbers if they are accepted or in a different juristiction this needs you to pay attention)
* Add to a collection
* Export the entire collection to .ris
* Import .ris into endnote

Sometimes, you'll have an issue where lens.org can't find the patent. For example, US2018030347A1 returns no hits. So in that case:

* Go to espacenet, enter the patent number.
* Examine the results. We find the patent is in other jurisdictions: EP3275971A1; EP3275971B1; JP2018016574A; JP6708042B2; US2018030347A1
* A code ending in "B" means its filed (I think), and I guess the higher the number the better right? So lets pick JP6708042B2
* Now on lens.org we search for this and find the patent "ジフルオロメチレンオキシ基を有する液晶性化合物、液晶組成物および液晶表示素子" - perfecto

But sometimes you won't find it on espacenet! And this is because of curation errors in the dataset (not everything typed is flawless). Now, your job is harder:
* Find an example structure in the dataset for the patent in question
* copy the smiles and paste into REAXYS and look for matching documents
* Locate the patent on espacenet and pick a patent number out, then use lens to get it added to the collection.

Ideally we could use the lens.org API but its _not free_. There is a free trial though.

In [2]:
import pandas as pd
import requests

In [20]:
file_path = 'new_DE_data.xlsx'
df= pd.read_excel(file_path)

clean_data = df.dropna(how="all", inplace=True)
pd.set_option('display.max_rows', 10)
#df

In [5]:
def get_ris_from_dois(input_data):
    if type(input_data) ==  pd.core.frame.DataFrame:
        dois = data["Reference"].dropna().unique()
    elif type(input_data) == list:
        dois = input_data
    elif type(input_data) == str:
        dois = input_data
        
    with open("references.ris", "w", encoding="utf-8") as f:
        for doi in dois:
            doi = doi.strip()
            if not doi.startswith("10."):
                if not doi.startswith("https."): 
                    if '/10.' in doi:
                        doi = '10.' + doi.split('/10.')[1]
                    else:
                        continue
                        
            url = f"https://doi.org/{doi}"
            headers = {"Accept": "application/x-research-info-systems"}
            try:
                response = requests.get(url, headers=headers)
                print(response)
                if response.status_code == 200:
                    f.write(response.text + "\n")
                    print(f"Retrieved: {doi}")
                    print(response.text)
                else:
                    print(f"Failed ({response.status_code}): {doi}")
            except Exception as e:
                print(f"Error with {doi}: {e}")

In [21]:
def unique_refs(df, ref_col = 'DOI'):
    papers = []
    patents = []
    references = list(df[ref_col].unique())
    for ref in references:
        if ref.startswith("10."):
            papers.append(ref)
            continue
        if ref.startswith("http"):
            papers.append(ref)
            continue
        else:
            patents.append(ref)
            continue
    return papers, patents
    
papers, patents = unique_refs(df, ref_col = 'DOI')
print(patents)

['WO2015190399A1\xa0', 'TWI629343B', 'WO2020026583A1', 'WO2016133035A1', 'US2016280997', 'WO2006125530A1', 'EP2690103', 'US2016032187', 'EP2351741', 'EP2116522', 'WO2007147516', 'US2011301360', 'US2016280996', 'US201385306', 'US2016186060', 'JP2016034933', 'EP1182186', 'EP2586764', 'EP2957554', 'EP2960226', 'JP5828781', 'CN103788039', 'JP201816574', 'US9303208', 'CN105481824', 'CN105669595', 'US9481828', 'CN104402683', 'CN117510315', 'US8304036', 'US7846514', 'US9777216', 'US6475595', 'US7744968', 'US7122228', 'US7087272', 'US7514127', 'US7732022', 'US5807500', 'US6268608', 'US20150259602', 'US7014890', 'US5725799', 'US6793984', 'US11261163', 'US6541081', 'US7604851', 'US7385067', 'US7531106', 'US7361388', 'US7344761', 'US20080020148', 'US7291367', 'US7270856', 'US7135210', 'US7022865', 'US6780477', 'US6723866', 'US6706338', 'US 6635318', 'US6620468', 'US6139773', 'US6056895', 'US5997766', 'US5961880', 'JP5850023', 'US2025361444', 'WO2025196107', 'CN117903814', 'US20250154412', 'US2025